[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abigailhaddad/fedscope_new/blob/main/demo.ipynb)

# EHRI Federal Workforce Data Explorer

This notebook loads federal workforce data from HuggingFace and lets you explore trends over time:
- **Accessions** - New federal hires
- **Separations** - Federal employee departures
- **Employment** - Point-in-time workforce snapshots

**No authentication required** - all datasets are public. Available months are discovered automatically.

In [ ]:
!pip install -q duckdb pandas plotly huggingface_hub

In [ ]:
import re
import duckdb
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from huggingface_hub import list_repo_files
import base64
from IPython.display import HTML, display

def download_csv(df, filename):
    csv = df.to_csv(index=False)
    b64 = base64.b64encode(csv.encode()).decode()
    display(HTML(f'<a href="data:file/csv;base64,{b64}" download="{filename}" style="background:#4CAF50;color:white;padding:8px 16px;text-decoration:none;border-radius:4px;display:inline-block;margin:10px 0;">Download {filename}</a>'))

## 1. Discover Available Files

We query the HuggingFace repo to find all available months automatically.

In [ ]:
HF_REPO = "abigailhaddad/opm-federal-workforce"
BASE_URL = f"https://huggingface.co/datasets/{HF_REPO}/resolve/main"

all_files = list(list_repo_files(HF_REPO, repo_type="dataset"))
parquet_files = [f for f in all_files if f.endswith(".parquet")]

def get_urls(data_type):
    """Get sorted list of HF URLs for a given data type."""
    pattern = re.compile(rf"^{data_type}/{data_type}_(\d{{6}})\.parquet$")
    matches = [(pattern.match(f).group(1), f) for f in parquet_files if pattern.match(f)]
    matches.sort()
    return [f"{BASE_URL}/{path}" for _, path in matches], [m for m, _ in matches]

acc_urls, acc_months = get_urls("accessions")
sep_urls, sep_months = get_urls("separations")
emp_urls, emp_months = get_urls("employment")

print(f"Accessions:  {len(acc_urls)} files  ({acc_months[0]} - {acc_months[-1]})")
print(f"Separations: {len(sep_urls)} files  ({sep_months[0]} - {sep_months[-1]})")
print(f"Employment:  {len(emp_urls)} files  ({emp_months[0]} - {emp_months[-1]})")

## 2. Load Accessions & Separations

These are small (~0.2 MB/month parquet) so we load all available months into DuckDB. Queries after this are instant.

In [ ]:
%%time
db = duckdb.connect()

acc_list = ", ".join(f"'{u}'" for u in acc_urls)
db.execute(f"CREATE TABLE accessions AS SELECT * FROM read_parquet([{acc_list}])")

sep_list = ", ".join(f"'{u}'" for u in sep_urls)
db.execute(f"CREATE TABLE separations AS SELECT * FROM read_parquet([{sep_list}])")

acc_count = db.execute("SELECT SUM(CAST(count AS INTEGER)) FROM accessions").fetchone()[0]
sep_count = db.execute("SELECT SUM(CAST(count AS INTEGER)) FROM separations").fetchone()[0]
print(f"Loaded {acc_count:,} accession records and {sep_count:,} separation records")

## 3. Accessions vs Separations Over Time

In [ ]:
acc_df = db.execute("""
    SELECT personnel_action_effective_date_yyyymm as month,
           SUM(CAST(count AS INTEGER)) as count
    FROM accessions
    GROUP BY month ORDER BY month
""").df()
acc_df['date'] = pd.to_datetime(acc_df['month'], format='%Y%m')

sep_df = db.execute("""
    SELECT personnel_action_effective_date_yyyymm as month,
           SUM(CAST(count AS INTEGER)) as count
    FROM separations
    GROUP BY month ORDER BY month
""").df()
sep_df['date'] = pd.to_datetime(sep_df['month'], format='%Y%m')

fig = go.Figure()
fig.add_trace(go.Scatter(x=acc_df['date'], y=acc_df['count'], mode='lines+markers',
    name='Accessions (New Hires)', line=dict(color='#2ecc71', width=2),
    hovertemplate='%{x|%b %Y}<br>%{y:,.0f} new hires<extra></extra>'))
fig.add_trace(go.Scatter(x=sep_df['date'], y=sep_df['count'], mode='lines+markers',
    name='Separations (Departures)', line=dict(color='#e74c3c', width=2),
    hovertemplate='%{x|%b %Y}<br>%{y:,.0f} departures<extra></extra>'))
fig.update_layout(title='Federal Workforce: Monthly Accessions vs Separations',
    xaxis_title='Month', yaxis_title='Count', hovermode='x unified',
    template='plotly_white', yaxis=dict(tickformat=','), height=500,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1))
fig.show()

merged = acc_df[['month','count']].rename(columns={'count':'accessions'}).merge(
    sep_df[['month','count']].rename(columns={'count':'separations'}), on='month')
download_csv(merged, 'monthly_accessions_separations.csv')

In [ ]:
merged['net_change'] = merged['accessions'] - merged['separations']
merged['date'] = pd.to_datetime(merged['month'], format='%Y%m')
merged['color'] = merged['net_change'].apply(lambda x: '#2ecc71' if x > 0 else '#e74c3c')

fig = go.Figure(go.Bar(x=merged['date'], y=merged['net_change'],
    marker_color=merged['color'],
    hovertemplate='%{x|%b %Y}<br>Net: %{y:+,.0f}<extra></extra>'))
fig.add_hline(y=0, line_width=1, line_color='black')
fig.update_layout(title='Federal Workforce: Monthly Net Change (Accessions - Separations)',
    xaxis_title='Month', yaxis_title='Net Change',
    template='plotly_white', yaxis=dict(tickformat=','), height=450, showlegend=False)
fig.show()

print(f"Total accessions: {merged['accessions'].sum():,}")
print(f"Total separations: {merged['separations'].sum():,}")
print(f"Net change: {merged['net_change'].sum():+,}")
download_csv(merged[['month','accessions','separations','net_change']], 'monthly_net_change.csv')

## 4. Biggest Hiring Agencies

In [ ]:
top_agencies = db.execute("""
    SELECT agency, SUM(CAST(count AS INTEGER)) as n
    FROM accessions GROUP BY agency ORDER BY n DESC LIMIT 6
""").df()['agency'].tolist()

fig = go.Figure()
frames = []
for agency in top_agencies:
    df = db.execute(f"""
        SELECT personnel_action_effective_date_yyyymm as month,
               SUM(CAST(count AS INTEGER)) as count
        FROM accessions WHERE agency = '{agency}'
        GROUP BY month ORDER BY month
    """).df()
    df['date'] = pd.to_datetime(df['month'], format='%Y%m')
    df['agency'] = agency
    frames.append(df[['month','agency','count']])
    fig.add_trace(go.Scatter(x=df['date'], y=df['count'], mode='lines',
        name=agency[:35], hovertemplate='%{x|%b %Y}<br>%{y:,.0f}<extra></extra>'))

fig.update_layout(title='Monthly Accessions by Top 6 Agencies',
    xaxis_title='Month', yaxis_title='New Hires', template='plotly_white', height=500,
    yaxis=dict(tickformat=','), legend=dict(orientation='h', yanchor='bottom', y=-0.3))
fig.show()
download_csv(pd.concat(frames), 'accessions_by_agency.csv')

## 5. Columns & Available Values

In [ ]:
print("Accessions columns:", db.execute("DESCRIBE accessions").df()['column_name'].tolist())
print("\nSeparations columns:", db.execute("DESCRIBE separations").df()['column_name'].tolist())

In [ ]:
for field in ['agency', 'occupational_group', 'education_level', 'duty_station_state']:
    result = db.execute(f"""
        SELECT {field} as value, SUM(CAST(count AS INTEGER)) as n
        FROM accessions WHERE {field} IS NOT NULL AND {field} != ''
        GROUP BY {field} ORDER BY n DESC LIMIT 8
    """).df()
    print(f"\n{field}:")
    for _, row in result.iterrows():
        print(f"  {row['value']}: {row['n']:,}")

## 6. Employment Data (Point-in-Time Snapshots)

Employment files are large (~30 MB/month parquet). We load only the most recent 6 months by default — adjust `N_MONTHS` as needed.

In [ ]:
%%time
N_MONTHS = 6
recent_emp_urls = emp_urls[-N_MONTHS:]
recent_emp_months = emp_months[-N_MONTHS:]
print(f"Loading employment: {recent_emp_months[0]} - {recent_emp_months[-1]}")

emp_list = ", ".join(f"'{u}'" for u in recent_emp_urls)
db.execute(f"CREATE OR REPLACE TABLE employment AS SELECT * FROM read_parquet([{emp_list}])")

emp_count = db.execute("SELECT SUM(CAST(count AS INTEGER)) FROM employment").fetchone()[0]
print(f"Loaded {emp_count:,} employee records")
print("Columns:", db.execute("DESCRIBE employment").df()['column_name'].tolist())

In [ ]:
# Total workforce over time
emp_total = db.execute("""
    SELECT snapshot_yyyymm as month, SUM(CAST(count AS INTEGER)) as employees
    FROM employment GROUP BY month ORDER BY month
""").df()
emp_total['date'] = pd.to_datetime(emp_total['month'], format='%Y%m')

fig = go.Figure(go.Scatter(x=emp_total['date'], y=emp_total['employees'],
    mode='lines+markers', line=dict(color='#3498db', width=2),
    hovertemplate='%{x|%b %Y}<br>%{y:,.0f} employees<extra></extra>'))
fig.update_layout(title='Total Federal Workforce (Employment Snapshots)',
    xaxis_title='Month', yaxis_title='Employees',
    template='plotly_white', height=400, yaxis=dict(tickformat=','))
fig.show()
download_csv(emp_total[['month','employees']], 'total_workforce.csv')

In [ ]:
# Workforce by top agencies
latest_month = emp_months[-1]
top_emp_agencies = db.execute(f"""
    SELECT agency, SUM(CAST(count AS INTEGER)) as n
    FROM employment WHERE snapshot_yyyymm = '{latest_month}'
    GROUP BY agency ORDER BY n DESC LIMIT 6
""").df()['agency'].tolist()

fig = go.Figure()
emp_frames = []
for agency in top_emp_agencies:
    df = db.execute(f"""
        SELECT snapshot_yyyymm as month, SUM(CAST(count AS INTEGER)) as employees
        FROM employment WHERE agency = '{agency}'
        GROUP BY month ORDER BY month
    """).df()
    df['date'] = pd.to_datetime(df['month'], format='%Y%m')
    df['agency'] = agency
    emp_frames.append(df[['month','agency','employees']])
    fig.add_trace(go.Scatter(x=df['date'], y=df['employees'], mode='lines+markers',
        name=agency[:35], hovertemplate='%{x|%b %Y}<br>%{y:,.0f}<extra></extra>'))

fig.update_layout(title=f'Workforce by Top 6 Agencies (recent {N_MONTHS} months)',
    xaxis_title='Month', yaxis_title='Employees', template='plotly_white', height=500,
    yaxis=dict(tickformat=','), legend=dict(orientation='h', yanchor='bottom', y=-0.35))
fig.show()
download_csv(pd.concat(emp_frames), 'workforce_by_agency.csv')

## 7. Try Your Own Queries

Three DuckDB tables are loaded: `accessions`, `separations`, `employment`.

**Common fields:**
- `agency` — e.g. `DEPARTMENT OF DEFENSE`
- `duty_station_state` — e.g. `CALIFORNIA`
- `occupational_group` — e.g. `INFORMATION TECHNOLOGY GROUP`
- `occupational_series` — e.g. `2210`
- `education_level`, `age_bracket`, `supervisory_status`
- `personnel_action_effective_date_yyyymm` (accessions/separations)
- `snapshot_yyyymm` (employment)
- `count` — number of people in that combination of values

All counts are stored as strings — cast with `CAST(count AS INTEGER)`.

In [ ]:
# Example: Army separations by month
df = db.execute("""
    SELECT personnel_action_effective_date_yyyymm as month,
           SUM(CAST(count AS INTEGER)) as departures
    FROM separations
    WHERE agency = 'DEPARTMENT OF THE ARMY'
    GROUP BY month ORDER BY month
""").df()
df['date'] = pd.to_datetime(df['month'], format='%Y%m')

fig = go.Figure(go.Scatter(x=df['date'], y=df['departures'], mode='lines+markers',
    line=dict(color='#e74c3c')))
fig.update_layout(title='Army Separations by Month', template='plotly_white',
    yaxis=dict(tickformat=','), height=400)
fig.show()